# Qwen-Image-2.1 免费 Colab｜GitHub 任务队列 V4（交互执行）

**工作方式**：你在 ChatGPT 发母图和中文修改词 → ChatGPT 在 `jobs/pending/` 提交任务 → 你打开此 Notebook 并点击运行，自动取当前任务、生成图片、上传到 `images/`，同时记录 `jobs/results/`。

**限制**：免费 Colab 不能作为长期后台接单 Worker；本 Notebook **不会**无休止轮询或远程接管 GPU。Google 官方 FAQ 限制免费版分布式计算 Worker，且 GPU 无法保证。输出结果可另由 ChatGPT 定期检查通知。

**仓库公开**：输入母图以 base64 文本储存在公开仓库，**base64 并非加密**。不要用隐私图片。操作说明见 [QUEUE_GUIDE.md](https://github.com/mjw3177109/qwen2.1tu/blob/main/QUEUE_GUIDE.md)。

In [ ]:
# 01 / 配置 + GPU、磁盘 + GitHub Token 预检查（先执行，避免白下载模型）
from pathlib import Path
import os, sys, shutil, subprocess, time, io, json, re, uuid, base64
import requests

REPO = "mjw3177109/qwen2.1tu"
BRANCH = "main"
EDIT_PROMPT = "<image1> 保持人物身份、姿态、背景与二维动漫风格不变，只把衣服变成深蓝色。"
EDIT_RESOLUTION = 512       # 先用 512；成功后可尝试 640 或 768
ENCODER_MODE = "w4a8"      # 小体积 6.31GB；不兼容可试 int8（更占磁盘）
SEED = 42
ROOT = Path("/content/qwen21_light")
COMFY = ROOT / "ComfyUI"
OUT = Path("/content/qwen21_outputs")
OUT.mkdir(parents=True, exist_ok=True)
(ROOT / "tmp").mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(ROOT / "hf_cache")
os.environ["HF_HUB_CACHE"] = str(ROOT / "hf_cache/hub")
os.environ["HF_XET_CACHE"] = str(ROOT / "hf_cache/xet")
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"
os.environ["PIP_NO_CACHE_DIR"] = "1"
os.environ["TMPDIR"] = str(ROOT / "tmp")
os.environ["XDG_CACHE_HOME"] = str(ROOT / "xdg_cache")

from google.colab import userdata
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise RuntimeError("请先在 Colab 左侧 🔑 Secrets 添加 GITHUB_TOKEN 并允许 Notebook 访问。")

GH_HEADERS = {
    "Authorization": "Bearer " + GITHUB_TOKEN,
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
probe = requests.get("https://api.github.com/repos/" + REPO, headers=GH_HEADERS, timeout=25)
if probe.status_code != 200:
    raise RuntimeError("GitHub 验证失败：HTTP " + str(probe.status_code) + "。检查 Token 是否授权此仓库。")
print("GitHub 仓库连接成功:", REPO)
if not probe.json().get("permissions", {}).get("push", False):
    print("⚠️ Token 未被确认具有 push 权限；请确保仓库 Contents 是 Read and write。")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("未检测到 GPU，请在运行时设置里选择 GPU。Colab 免费额度可能暂不可用。")
for i in range(torch.cuda.device_count()):
    prop = torch.cuda.get_device_properties(i)
    print("GPU:", prop.name, f"{prop.total_memory/2**30:.1f} GiB")
free = shutil.disk_usage("/content").free
print("Colab 临时磁盘剩余:", round(free/2**30, 1), "GiB")
if free < 17 * 2**30:
    raise RuntimeError("临时磁盘不足 17 GiB，停止以避免下载失败。")
try:
    import psutil
    print("系统总 RAM:", round(psutil.virtual_memory().total/2**30, 1), "GiB")
except ImportError:
    pass
assert ENCODER_MODE in ("w4a8", "int8")

In [ ]:
# 02 / 安装 ComfyUI + GGUF 扩展（复用 Colab 自带 torch）
def run_cmd(args, cwd=None):
    subprocess.run(list(map(str, args)), cwd=str(cwd) if cwd else None, check=True)

if not (COMFY / "main.py").exists():
    run_cmd(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", COMFY])
node_dir = COMFY / "custom_nodes" / "ComfyUI-GGUF"
if not (node_dir / "nodes.py").exists():
    run_cmd(["git", "clone", "--depth", "1", "https://github.com/city96/ComfyUI-GGUF.git", node_dir])

requirements = []
for line in (COMFY / "requirements.txt").read_text(encoding="utf-8").splitlines():
    s = line.strip()
    if s and not s.startswith("#") and not re.match(r"^(torch|torchvision|torchaudio)(\b|[<>=~!\[])", s, flags=re.I):
        requirements.append(line)
req = ROOT / "requirements_without_torch.txt"
req.write_text("\n".join(requirements) + "\n", encoding="utf-8")
run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "-r", req])
run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "huggingface_hub>=0.33", "gguf", "requests"])
gguf_req = node_dir / "requirements.txt"
if gguf_req.exists():
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "-r", gguf_req])
print("ComfyUI 环境安装完成。")

In [ ]:
# 03 / 仅下载图片编辑需要的 3 个权重（模型存 Colab 临时磁盘，不保存到 GitHub）
from huggingface_hub import hf_hub_download

MODELS = COMFY / "models"
GGUF_FILE = "qwen-image-2.1-turbo-Q4_K_M.gguf"
ENC_FILE = ("qwen3vl_8b_w4a8.safetensors" if ENCODER_MODE == "w4a8"
            else "qwen3vl_8b_int8_convrot.safetensors")
VAE_FILE = "qwen_image_2.1_vae_bf16.safetensors"

weights = [
    ("abenzerps/Qwen-Image-2.1-Turbo-Quantized", GGUF_FILE, MODELS/"diffusion_models", 4.05),
    ("Comfy-Org/Qwen-Image-2.1", "text_encoders/"+ENC_FILE, MODELS, 6.31 if ENCODER_MODE=="w4a8" else 8.71),
    ("Comfy-Org/Qwen-Image-2.1", "vae/"+VAE_FILE, MODELS, 0.65),
]
for repo_id, filename, folder, size_gb in weights:
    folder.mkdir(parents=True, exist_ok=True)
    target = folder/filename
    if target.is_file() and target.stat().st_size > size_gb * 1e9 * 0.90:
        print("已存在，跳过:", target.name)
        continue
    if shutil.disk_usage("/content").free < size_gb*1e9 + 2e9:
        raise RuntimeError("磁盘不足，未下载 " + filename)
    print("下载:", repo_id, filename)
    hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(folder))
    if not target.is_file():
        raise FileNotFoundError("下载后找不到模型: " + str(target))
    print("完成:", target.name, round(target.stat().st_size/1e9, 2), "GB")
print("三个必需模型下载完成。")

In [ ]:
# 04 / 启动本地 ComfyUI 并检查所需节点
LOG = ROOT / "comfyui.log"
URL = "http://127.0.0.1:8188"
def healthy():
    try:
        return requests.get(URL+"/system_stats", timeout=3).ok
    except requests.RequestException:
        return False

if not healthy():
    cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
           "--lowvram", "--disable-auto-launch", "--output-directory", str(OUT),
           "--temp-directory", str(ROOT/"tmp"), "--input-directory", str(COMFY/"input")]
    with LOG.open("a", encoding="utf-8") as f:
        comfy_process = subprocess.Popen(cmd, cwd=COMFY, stdout=f, stderr=subprocess.STDOUT)
    for _ in range(90):
        if healthy():
            break
        if comfy_process.poll() is not None:
            break
        time.sleep(2)
if not healthy():
    print("\n".join(LOG.read_text(errors="replace").splitlines()[-90:]))
    raise RuntimeError("ComfyUI 启动失败")

infos = requests.get(URL+"/object_info", timeout=45).json()
needed = ("UnetLoaderGGUF", "CLIPLoader", "VAELoader", "LoadImage",
          "TextEncodeQwenImage21", "KSamplerSelect", "ManualSigmas",
          "SamplerCustom", "VAEDecode", "SaveImage")
missing = [node for node in needed if node not in infos]
if missing:
    raise RuntimeError("当前 ComfyUI 版本缺少节点: " + ", ".join(missing))
print("编辑节点全部存在，ComfyUI 已就绪。")

In [ ]:
# 06 / 图像编辑工作流（参考图确实接入 Qwen-Image-2.1 节点）
import secrets
SIGMAS = "1.0, 0.978453, 0.95418, 0.926626, 0.89508, 0.845148, 0.704534, 0.414568, 0.0"

def make_graph(prompt, input_name, seed):
    return {
        "1": {"class_type":"UnetLoaderGGUF","inputs":{"unet_name":GGUF_FILE}},
        "2": {"class_type":"CLIPLoader","inputs":{"clip_name":ENC_FILE, "type":"qwen_image", "device":"default"}},
        "3": {"class_type":"VAELoader","inputs":{"vae_name":VAE_FILE}},
        "4": {"class_type":"LoadImage","inputs":{"image":input_name}},
        "5": {"class_type":"TextEncodeQwenImage21","inputs":{
            "clip":["2",0], "vae":["3",0], "prompt":prompt, "negative_prompt":"",
            "resolution":EDIT_RESOLUTION, "images.image_1":["4",0]}},
        "6": {"class_type":"KSamplerSelect","inputs":{"sampler_name":"euler"}},
        "7": {"class_type":"ManualSigmas","inputs":{"sigmas":SIGMAS}},
        "8": {"class_type":"SamplerCustom","inputs":{
            "model":["1",0], "add_noise":True, "noise_seed":int(seed),
            "cfg":1.0, "positive":["5",0], "negative":["5",1],
            "sampler":["6",0], "sigmas":["7",0], "latent_image":["5",2]}},
        "9": {"class_type":"VAEDecode","inputs":{"samples":["8",0], "vae":["3",0]}},
        "10":{"class_type":"SaveImage","inputs":{
            "images":["9",0], "filename_prefix":"Qwen21_Edit_Github"}}
    }

# 校验参考图连接，防止误生成无关图片
assert make_graph("test", INPUT_NAME, SEED)["5"]["inputs"]["images.image_1"] == ["4",0]
print("修改提示词:", EDIT_PROMPT)

## 母图上传与任务入队（仅首次上传原图）

这里从 GitHub 读取我提交的优化提示词，并让你选择最初的母图。它会核验 SHA256，确认**确实是聊天中给出的母图**才加入待办队列，避免误用聊天中的 AI 生成图。上传一次后即可运行下方任务执行单元。源图与成品均提交到公开仓库，请勿用于隐私图片。

In [ ]:
# 06 / 将 ChatGPT 已提交的优化提示词 + 原始母图组成正式待办任务
# 首次需要在这里上传与聊天中相同的母图，以后同理；图片不会由 ChatGPT 直接生图。
from google.colab import files as colab_files
from IPython.display import display
from PIL import Image
from pathlib import Path
import io, json, hashlib, base64, requests, re

DRAFT_JOB_ID = "edit_20261019_yellowhelmet01"
GH_BASE = f"https://api.github.com/repos/{REPO}/contents"

def get_repo_item(path):
    res = requests.get(f"{GH_BASE}/{path}",headers=GH_HEADERS,params={"ref":BRANCH},timeout=30)
    if res.status_code==404: return None
    res.raise_for_status()
    return res.json()

def get_repo_text(path):
    data=get_repo_item(path)
    if not data: raise FileNotFoundError("GitHub 上不存在："+path)
    return base64.b64decode(data["content"]).decode("utf-8-sig")

def create_repo_text_if_missing(path,content):
    old=get_repo_item(path)
    if old: return "EXISTS"
    res=requests.put(f"{GH_BASE}/{path}",headers=GH_HEADERS,
        json={"message":f"jobs: add {path}","content":base64.b64encode(content.encode("utf-8")).decode("ascii"),"branch":BRANCH},
        timeout=120)
    if res.status_code not in (200,201):
        raise RuntimeError(f"Github 创建失败：{path} HTTP {res.status_code}: {res.text[:400]}")
    verify=get_repo_item(path)
    if not verify or verify.get("sha")!=res.json().get("content",{}).get("sha"):
        raise RuntimeError("创建后回读失败："+path)
    return "CREATED"

if not re.fullmatch(r"[a-zA-Z0-9_-]{8,64}",DRAFT_JOB_ID):
    raise ValueError("无效任务 ID")
draft=json.loads(get_repo_text(f"jobs/drafts/{DRAFT_JOB_ID}.json"))
assert draft["id"]==DRAFT_JOB_ID
assert draft["status"]=="WAITING_FOR_SOURCE_UPLOAD"
pending_path=f"jobs/pending/{DRAFT_JOB_ID}.json"
if get_repo_item(pending_path):
    print("该任务已进入待办队列，无需再次上传。")
else:
    print("请上传聊天里那张原始母图，文件名：",draft["original_source_name"])
    uploaded=colab_files.upload()
    if len(uploaded)!=1: raise ValueError("只允许上传 1 张原始母图")
    filename,raw=next(iter(uploaded.items()))
    if Path(filename).suffix.lower() not in (".jpg",".jpeg",".png",".webp"):
        raise ValueError("母图必须是 JPG/PNG/WEBP")
    if len(raw)>4_000_000: raise ValueError("母图须小于 4MB，当前图片过大")
    with Image.open(io.BytesIO(raw)) as im:
        im.verify()
    digest=hashlib.sha256(raw).hexdigest()
    if digest != draft["expected_source_sha256"]:
        raise ValueError("原图 SHA256 与聊天中的母图不一致！请下载并上传原始附件，切勿用 AI 已修改图。")
    src_path=draft["input_b64_path"]
    print("原图 SHA256 验证通过，正在写入 GitHub 输入目录……")
    status=create_repo_text_if_missing(src_path,base64.b64encode(raw).decode("ascii"))
    prompt=draft["prompt"].strip()
    job={
        "schema_version":1,
        "id":DRAFT_JOB_ID,
        "kind":"image_edit",
        "input_b64_path":src_path,
        "input_sha256":digest,
        "prompt":prompt,
        "resolution":int(draft.get("resolution",512)),
        "seed":int(draft.get("seed",42)),
        "created_at":draft["created_at"]
    }
    # 源图已上传后才进入 pending，不把半成品任务交给 worker。
    create_repo_text_if_missing(pending_path,json.dumps(job,ensure_ascii=False,indent=2)+"\n")
    print("✅ 图片和提示词已合并成待办任务：",pending_path)
print("下一单元执行 GPU 编辑，并上传成品 PNG。")


## 处理待办任务

前面模型已加载并且 ComfyUI 正常启动后，运行下一单元。它只处理当前已提交的最多 3 个任务，生成成功后**自动上传成品 PNG、保存结果 JSON**；失败时保存失败原因。下一次有新的母图和提示词，再运行这一个单元即可。

请勿关闭 Colab 或断开 GPU 直到本次任务结束。

In [ ]:
# 06 / 从 GitHub 待办目录一次性执行当前任务（非后台常驻 Worker）
# Colab 免费版请主动点击本单元；只处理当前队列，完成后结束。
import requests, io, json, base64, hashlib, uuid, time, re
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image, ImageOps
from IPython.display import display
GH_BASE = f"https://api.github.com/repos/{REPO}/contents"
MAX_JOBS_PER_RUN = 3
JOB_TIMEOUT_SECONDS = 2400
MAX_SOURCE_BYTES = 4_000_000

def now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00","Z")

def gh_read(path, missing_ok=False):
    r = requests.get(f"{GH_BASE}/{path}", headers=GH_HEADERS,
                     params={"ref":BRANCH}, timeout=40)
    if r.status_code == 404 and missing_ok:
        return None
    r.raise_for_status()
    return r.json()

def gh_read_text(path, missing_ok=False):
    data = gh_read(path, missing_ok=missing_ok)
    if data is None:
        return None, None
    if data.get("type") != "file" or data.get("encoding") != "base64":
        raise RuntimeError("不支持的 GitHub 文件响应：" + path)
    text_bytes = base64.b64decode(data["content"])
    return text_bytes.decode("utf-8-sig"), data.get("sha")

def gh_write_bytes(path, body: bytes, message: str, existing_sha=None):
    payload = {"message":message, "content":base64.b64encode(body).decode("ascii"),
               "branch":BRANCH}
    if existing_sha:
        payload["sha"] = existing_sha
    r = requests.put(f"{GH_BASE}/{path}", headers=GH_HEADERS, json=payload,timeout=120)
    if r.status_code not in (200,201):
        # Token 与请求头绝不输出
        raise RuntimeError(f"Github 提交失败 {path} HTTP {r.status_code}: {r.text[:350]}")
    expected_sha = r.json().get("content",{}).get("sha")
    remote = gh_read(path)
    if remote.get("sha") != expected_sha:
        raise RuntimeError("GitHub 文件写入后核验失败：" + path)
    return expected_sha

def gh_write_json(path, obj, existing_sha=None):
    body = (json.dumps(obj,ensure_ascii=False,indent=2)+"\n").encode("utf-8")
    return gh_write_bytes(path, body, f"jobs: record {path}", existing_sha)

def append_event(event):
    path = "queue/events.json"
    for attempt in range(5):
        body, sha = gh_read_text(path, missing_ok=True)
        data = json.loads(body) if body else {"schema_version":1,"events":[]}
        arr = data["events"]
        if not any(x.get("job_id")==event["job_id"] for x in arr):
            arr.append(event)
        data["events"] = arr[-200:]
        try:
            gh_write_json(path,data,sha)
            return
        except RuntimeError as err:
            if "HTTP 409" not in str(err) and "HTTP 422" not in str(err):
                raise
            time.sleep(1+attempt)
    raise RuntimeError("通知事件更新冲突，请人工检查 GitHub queue/events.json")

def list_pending_jobs():
    r = gh_read("jobs/pending",missing_ok=True)
    if r is None:
        return []
    if not isinstance(r,list):
        raise RuntimeError("jobs/pending 并非目录")
    pending = []
    for item in r:
        if item.get("type")=="file" and item["name"].endswith(".json"):
            txt,_=gh_read_text(item["path"])
            pending.append(json.loads(txt))
    return sorted(pending,key=lambda j:(j.get("created_at",""),j.get("id","")))

def validate_job(job):
    job_id = str(job.get("id",""))
    if not re.fullmatch(r"[a-zA-Z0-9_-]{8,64}", job_id):
        raise ValueError("任务 ID 不合法")
    if job.get("schema_version") != 1 or job.get("kind")!="image_edit":
        raise ValueError("任务 schema/kind 不支持")
    src = job.get("input_b64_path","")
    if src not in (f"jobs/inputs/{job_id}.jpg.b64",
                   f"jobs/inputs/{job_id}.jpeg.b64",
                   f"jobs/inputs/{job_id}.png.b64",
                   f"jobs/inputs/{job_id}.webp.b64"):
        raise ValueError("母图路径与 job ID 不匹配")
    sha = str(job.get("input_sha256",""))
    if not re.fullmatch(r"[a-f0-9]{64}",sha):
        raise ValueError("缺少母图 sha256")
    prompt = str(job.get("prompt","")).strip()
    if not 3 <= len(prompt) <= 2500:
        raise ValueError("编辑提示词为空或过长")
    resolution = int(job.get("resolution",512))
    if resolution not in (512,640,768):
        raise ValueError("只支持分辨率 512/640/768")
    seed = int(job.get("seed",42))
    if not 0 <= seed < 2**64:
        raise ValueError("seed 不合法")
    return job_id,src,sha,prompt,resolution,seed

def prepare_image(job_id, source_path, expected_sha, resolution):
    b64_txt,_ = gh_read_text(source_path)
    raw = base64.b64decode("".join(b64_txt.split()),validate=True)
    if len(raw)>MAX_SOURCE_BYTES:
        raise ValueError("母图压缩体积超过 4MB")
    actual_sha = hashlib.sha256(raw).hexdigest()
    if actual_sha != expected_sha:
        raise ValueError("母图 SHA256 不匹配")
    Image.MAX_IMAGE_PIXELS = 48_000_000
    image = ImageOps.exif_transpose(Image.open(io.BytesIO(raw))).convert("RGB")
    if image.width*image.height>48_000_000:
        raise ValueError("原图像素过大")
    cap=resolution**2
    if image.width*image.height>cap:
        ratio=(cap/(image.width*image.height))**0.5
        w=max(32,round(image.width*ratio/32)*32)
        h=max(32,round(image.height*ratio/32)*32)
        image=image.resize((w,h), Image.Resampling.LANCZOS)
    name=f"queue_{job_id}.png"
    (COMFY/"input").mkdir(exist_ok=True,parents=True)
    image.save(COMFY/"input"/name)
    print("母图:",job_id,"实际编辑尺寸:",image.size)
    return name

def generate_one(job_id, prompt, source_name, seed, resolution):
    global EDIT_RESOLUTION
    EDIT_RESOLUTION=resolution
    graph=make_graph(prompt,source_name,seed)
    graph["10"]["inputs"]["filename_prefix"]=f"Qwen21_{job_id}"
    assert graph["5"]["inputs"]["images.image_1"]==["4",0], "未连接参考图"
    resp=requests.post(URL+"/prompt",
                       json={"prompt":graph,"client_id":uuid.uuid4().hex},timeout=75)
    if not resp.ok:
        raise RuntimeError(f"ComfyUI 工作流校验失败: {resp.text[:950]}")
    body=resp.json()
    if body.get("node_errors"):
        raise RuntimeError("ComfyUI 节点错误："+str(body["node_errors"])[:800])
    prompt_id=body["prompt_id"]
    deadline=time.monotonic()+JOB_TIMEOUT_SECONDS
    while time.monotonic()<deadline:
        history=requests.get(URL+"/history/"+prompt_id,timeout=30).json()
        entry=history.get(prompt_id)
        if entry:
            status=entry.get("status",{})
            if not status.get("completed",False):
                raise RuntimeError("ComfyUI 编辑失败："+str(status)[:750])
            images=entry.get("outputs",{}).get("10",{}).get("images",[])
            if not images:
                raise RuntimeError("任务结束但 SaveImage 无输出")
            info=images[0]
            path=OUT/info.get("subfolder","")/info["filename"]
            if not path.is_file():
                raise FileNotFoundError(str(path))
            return path
        time.sleep(3)
    raise TimeoutError("生成超时，任务仍可能在 ComfyUI 中执行，请检查当前运行时")

def execute_one(job):
    job_id,src,sha,prompt,res,seed=validate_job(job)
    result_path=f"jobs/results/{job_id}.json"
    if gh_read(result_path,missing_ok=True):
        print(f"⏭ 跳过已有结果: {job_id}")
        return "SKIPPED"
    image_path=f"images/{job_id}.png"
    # 在图片已提交、结果尚未写入的中断场景中复用原图（幂等）
    if not gh_read(image_path,missing_ok=True):
        source_name=prepare_image(job_id,src,sha,res)
        output_file=generate_one(job_id,prompt,source_name,seed,res)
        with Image.open(output_file) as output_image:
            output_image.verify()
        gh_write_bytes(image_path,output_file.read_bytes(),f"images: Qwen edit {job_id}")
        display(Image.open(output_file))
    image_url=f"https://github.com/{REPO}/blob/{BRANCH}/{image_path}"
    result={"schema_version":1,"job_id":job_id,"status":"SUCCEEDED",
            "input_path":src,"image_url":image_url,"finished_at":now_iso(),
            "seed":seed,"resolution":res}
    gh_write_json(result_path,result)
    event={"job_id":job_id,"status":"SUCCEEDED",
           "image_url":image_url,"finished_at":result["finished_at"]}
    append_event(event)
    print("✅ 成功：",image_url)
    return "SUCCEEDED"

def execute_pending(max_jobs=MAX_JOBS_PER_RUN):
    if not healthy():
        raise RuntimeError("ComfyUI 未运行，请先运行前面的启动单元")
    tasks=list_pending_jobs()
    print("GitHub 队列任务总数:",len(tasks))
    ran=0
    for job in tasks:
        if ran>=max_jobs: break
        job_id=str(job.get("id","UNKNOWN"))
        if gh_read(f"jobs/results/{job_id}.json",missing_ok=True):
            continue
        ran+=1
        try:
            print(f"\n▶️ 执行任务 {job_id} ({ran}/{max_jobs})")
            execute_one(job)
        except KeyboardInterrupt:
            print("执行已人工中断，未标记为失败，可重新启动。")
            raise
        except Exception as exc:
            print("❌ 任务失败:",job_id,type(exc).__name__,str(exc)[:600])
            if re.fullmatch(r"[A-Za-z0-9_-]{8,64}",job_id):
                failed={"schema_version":1,"job_id":job_id,"status":"FAILED",
                        "error":f"{type(exc).__name__}: {str(exc)[:500]}",
                        "finished_at":now_iso()}
                try:
                    if not gh_read(f"jobs/results/{job_id}.json",missing_ok=True):
                        gh_write_json(f"jobs/results/{job_id}.json",failed)
                    append_event({"job_id":job_id,"status":"FAILED",
                                  "finished_at":failed["finished_at"]})
                except Exception as log_exc:
                    print("记录失败信息也失败，请检查仓库：",type(log_exc).__name__)
    if ran==0:
        print("当前没有新任务。你发来母图和提示词，我提交任务后再运行本单元即可。")
    else:
        print(f"本次处理 {ran} 个任务；结果在 GitHub jobs/results 和 images 中。")
    print("本单元执行完成，未启用后台轮询。")

execute_pending()


## 查看成果与失败记录

- [生成图片目录](https://github.com/mjw3177109/qwen2.1tu/tree/main/images)
- [任务结果目录](https://github.com/mjw3177109/qwen2.1tu/tree/main/jobs/results)
- [最新通知事件](https://github.com/mjw3177109/qwen2.1tu/blob/main/queue/events.json)

如果遇到显存、依赖或节点错误，在 ChatGPT 发对应结果 JSON / 错误日志即可定位。当前模型执行仍待真实 GPU 验收。